# UAIA — full-data judge on A100

**One complete training run. No data-size or learning-rate sweep.**

1. Select **Runtime → Change runtime type → A100 GPU** and save.
2. Run the cells below in order. Upload **Judge_A100_FullData_Package.zip** when asked.
3. Authorize your Google Drive mount. Then run the **Start the full run** cell once.

The code checks the assigned GPU and stops if it is not an A100. It trains
Qwen3-4B-Instruct-2507 on **all 4,646 eligible clear pairs**, in both answer
orders: **9,292 presentations, 1,162 optimizer updates, one complete pass**.
Four pairs form an update; these small batches are normal training, not
separate experiments or manual training chunks.

A separate **400-pair monitor** selects a checkpoint. The 1,452 aggregate-tie
training pairs are excluded from this hard-label experiment. Afterward,
unchanged Qwen and the selected checkpoint each score all **396 validation
pairs twice** (311 clear labels, 85 ties diagnostic). Original model weights
remain frozen; only the LoRA additions learn. This is judge fine-tuning,
not foundation-model pretraining or product integration.

The automatic sequence has a **six-hour compute ceiling**, not an ETA:
up to 20 minutes for the GPU check, 300 for training and 20 per validation,
also bounded by remaining total time. Setup is separately capped at 20
minutes. A100 runtime has not yet been measured for this recipe. Checkpoints
and reports go to Drive; Colab can still end a session. GPU availability
depends on Colab: [official FAQ](https://research.google.com/colaboratory/faq.html).

The isolated environment avoids the earlier image/audio package conflict
and does **not** require restarting the notebook kernel. No calibration or
final-test data is included. NVIDIA HelpSteer2 data: CC BY 4.0; Qwen model:
Apache-2.0. Exact revisions and credits are in the package README.


In [ ]:
from pathlib import Path
import hashlib, json, zipfile
root = Path('/content/judge_a100_full').resolve()
expected_archive_sha256 = '2657c35f07efb6bc911bd470c5a56b805ee69876be9a2a4b1c92f4d50f434d1a'
if not root.exists():
    archive_path = Path('/content/Judge_A100_FullData_Package.zip')
    if not archive_path.exists():
        from google.colab import files
        uploaded = files.upload()
        assert len(uploaded) == 1, 'Upload exactly one Judge_A100_FullData_Package.zip.'
        archive_path = Path(next(iter(uploaded)))
        del uploaded
    assert hashlib.sha256(archive_path.read_bytes()).hexdigest() == expected_archive_sha256, 'Wrong package: use the ZIP supplied with this notebook.'
    with zipfile.ZipFile(archive_path) as archive:
        names = archive.namelist()
        assert len(names) == len(set(names)), 'Duplicate archive entries.'
        for name in names:
            assert (root / name).resolve().is_relative_to(root), 'Unsafe archive path.'
        archive.extractall(root)
manifest = json.loads((root / 'package_manifest.json').read_text())
for name, digest in manifest.items():
    path = (root / name).resolve()
    assert path.is_relative_to(root) and hashlib.sha256(path.read_bytes()).hexdigest() == digest, 'Changed package file: ' + name
print('Verified', len(manifest), 'package files. Existing intact extraction can be reused.')


## Set up the isolated training environment

This cell checks for A100 before installing the pinned packages into a
separate environment. It leaves Colab's notebook Python and its installed
image/audio packages alone. It uses pip's documented
[separate-interpreter support](https://pip.pypa.io/en/stable/topics/python-option/).
Run this once; an intact completed setup is reused after its fingerprint check.


In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, time
root = Path('/content/judge_a100_full')
code_dir, bundle = root / 'code', root / 'bundle'
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], check=True, capture_output=True, text=True, timeout=30).stdout.strip()
assert 'A100' in gpu.upper(), 'Choose A100 in Runtime > Change runtime type; assigned GPU: ' + gpu
env_dir = Path('/content/judge_a100_env')
worker_python = env_dir / 'bin/python'
setup_receipt = root / 'isolated_setup.json'
requirements_hash = hashlib.sha256((code_dir / 'requirements.txt').read_bytes()).hexdigest()
if setup_receipt.exists():
    assert worker_python.exists() and json.loads(setup_receipt.read_text())['requirements_sha256'] == requirements_hash, 'Existing setup differs; inspect before replacing it.'
    print('Reusing the isolated environment.')
else:
    setup_started = time.monotonic()
    if not worker_python.exists():
        subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(env_dir)], check=True, timeout=120)
    command = [sys.executable, '-m', 'pip', '--python', str(worker_python), 'install', '--disable-pip-version-check', '--retries', '0', '--timeout', '60', '-r', str(code_dir / 'requirements.txt')]
    remaining = 1200 - (time.monotonic() - setup_started)
    assert remaining > 0, 'Setup budget exhausted.'
    with (root / 'setup.log').open('a') as log:
        installed = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT, timeout=remaining)
    print((root / 'setup.log').read_text()[-6000:])
    installed.check_returncode()
    setup_receipt.write_text(json.dumps({'requirements_sha256': requirements_hash, 'worker_python': str(worker_python), 'seconds': time.monotonic() - setup_started}, indent=2))
print('Assigned GPU:', gpu, '\nTraining Python:', worker_python, '\nNo notebook restart needed.')


## Run checks, then connect Drive

The tests include full-data coverage and an exact saved-state restart.
A failed check stops before training. Google authorization must complete
before checkpoints can be saved. If an embedded browser cannot complete
authorization, open this same notebook in regular Chrome for the mount.


In [ ]:
from pathlib import Path
import subprocess, json
root = Path('/content/judge_a100_full')
code_dir, bundle = root / 'code', root / 'bundle'
worker_python = Path('/content/judge_a100_env/bin/python')
check = subprocess.run([str(worker_python), '-m', 'unittest', 'discover', '-s', str(code_dir), '-p', 'test*.py'], capture_output=True, text=True, timeout=300)
(root / 'preflight_tests.txt').write_text(check.stdout + check.stderr)
print(check.stdout, end='')
print(check.stderr, end='')
check.check_returncode()
verify = "import sys; from full_run import require_a100, check_data; from pathlib import Path; assert sys.prefix != sys.base_prefix; print(require_a100()); print('Clear pairs:', check_data(Path(sys.argv[1]))[1])"
subprocess.run([str(worker_python), '-c', verify, str(bundle)], cwd=code_dir, check=True, timeout=60)
from google.colab import drive
drive.mount('/content/drive')
assert Path('/content/drive/MyDrive').is_dir(), 'Drive is not mounted.'
run_root = Path('/content/drive/MyDrive/judge_a100_full_20261007')
print('Durable output:', run_root)
print('Existing output:', run_root.exists(), '(the launch cell refuses to overwrite it)')


## Start the full run — execute this cell once

This automatically runs **GPU check → one full training pass → unchanged-model
validation → selected-checkpoint validation**. No further training cells or
approval switches are required. A saved checkpoint is a backup while the
same training process continues; it does not restart or split training.

Defaults stay fixed: learning rate 0.00002, JSON, hard labels, all 36 LoRA
layers, rank 16, clipping 1.0 and no length/strength weighting. Early stopping
is disabled for this one pass, so every eligible pair is read. Saving and
monitoring occur at updates 100, 200, …, 1100 and 1162. Allow space on Drive
for 12 adapter/optimizer checkpoints (several GB); old run folders are preserved.

The cell displays progress every 15 seconds. Interrupting it terminates its
process group and preserves saved evidence. A failure is reported without
an automatic retry. After a disconnect, inspect the existing run first;
rerunning this cell will not silently start a duplicate or resume a failed job.


In [ ]:
from pathlib import Path
import json, os, signal, subprocess, time
from IPython.display import clear_output
root = Path('/content/judge_a100_full')
code_dir, bundle = root / 'code', root / 'bundle'
worker_python = Path('/content/judge_a100_env/bin/python')
run_root = Path('/content/drive/MyDrive/judge_a100_full_20261007')
assert Path('/content/drive/MyDrive').is_dir(), 'Mount Drive first.'
assert not run_root.exists(), 'An output folder exists. Inspect it; do not launch a duplicate.'
supervisor_log = run_root.with_suffix('.supervisor.log')
launch_receipt = run_root.with_suffix('.launch.json')
assert not supervisor_log.exists() and not launch_receipt.exists(), 'A launch was already recorded. Inspect its process and evidence.'
command = [str(worker_python), '-u', str(code_dir / 'full_run.py'), '--bundle', str(bundle), '--output', str(run_root), '--minutes', '360']
started = time.monotonic()
with supervisor_log.open('x') as log:
    process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
with launch_receipt.open('x') as stream:
    json.dump({'pid': process.pid, 'command': command, 'compute_minutes': 360, 'automatic_retries': 0}, stream, indent=2)
try:
    while process.poll() is None:
        if time.monotonic() - started > 360 * 60 + 30:
            raise TimeoutError('Overall compute ceiling reached.')
        clear_output(wait=True)
        print('A100 FULL-DATA RUN | elapsed minutes:', round((time.monotonic() - started) / 60, 1))
        for name in ('status.json', 'training/progress.json', 'baseline_a100/progress.json', 'selected_a100/progress.json'):
            path = run_root / name
            if path.exists():
                try:
                    print(name, json.loads(path.read_text()))
                except (OSError, json.JSONDecodeError):
                    print(name, 'temporarily unreadable; inspect the process/log before diagnosing failure')
        print('Log:', supervisor_log)
        time.sleep(15)
    print(supervisor_log.read_text()[-8000:])
    if process.returncode != 0:
        raise RuntimeError('The run stopped unsuccessfully. Logs and checkpoints were preserved; no retry was started.')
except BaseException:
    if process.poll() is None:
        os.killpg(process.pid, signal.SIGTERM)
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
    raise
assert json.loads((run_root / 'status.json').read_text())['status'] == 'completed'
print('Completed. Reports and checkpoints are saved on Drive.')


## Read the result

Accuracy means agreement with the 311 clear human preference labels.
Log-loss measures probability error (lower is better); AUC measures ranking
quality (higher is better). The reports also contain uncertainty, paired
tests, shorter-winner/strength slices, answer-order agreement and all 85 ties.

Both the **final checkpoint that saw every pair** and the **best monitor
checkpoint** are retained. If the monitor prefers an earlier version, its
actual exposure is shown below. The previous 600-pair L4 judge reached 81.0%;
this run has no result until it completes. Its matching A100 baseline is
remeasured, and neither result is a calibrated final-test score.


In [ ]:
from pathlib import Path
import json
from IPython.display import Markdown, display
run_root = Path('/content/drive/MyDrive/judge_a100_full_20261007')
assert (run_root / 'summary.json').exists(), 'No completed result yet. Inspect status.json and the log.'
summary = json.loads((run_root / 'summary.json').read_text())
selection = json.loads((run_root / 'selection.json').read_text())
rows = []
for folder, label in (('baseline_a100', 'Unchanged Qwen on A100'), ('selected_a100', 'Selected trained judge')):
    report = json.loads((run_root / folder / 'full_report.json').read_text())
    result = report['views']['averaged']
    m = result['metrics']
    rows.append(f"| {label} | {result['matches']}/311 | {m['accuracy']['value']:.1%} | {m['log_loss']['value']:.3f} | {m['auc']['value']:.3f} |")
display(Markdown('| Model | Clear matches | Accuracy | Log-loss | AUC |\n| --- | ---: | ---: | ---: | ---: |\n' + '\n'.join(rows)))
print('Full training pass:', summary['training_pairs'], 'pairs;', summary['updates'], 'updates')
print('Selected checkpoint saw:', selection['selected_checkpoint_training_pairs_seen'], 'pairs')
print('Selected:', selection['checkpoint'])
print('Final all-data checkpoint:', selection['final_checkpoint'])
print('Development screen:', summary['development_screen'])
print('Reports:', run_root / 'selected_a100/full_report.json')
print('Calibration, final test and product integration have not run. Disconnect the GPU when finished.')
